In [ ]:
from perfetto.trace_processor import TraceProcessor

In [ ]:
class LBO(object):
    GC_THREAD = "HeapTaskDaemon"

    def __init__(self, trace_path: str):
        self.tp = TraceProcessor(trace=trace_path)
    
    def __get_cpu_time(self, process: str, filter = None) -> int:
        process = next(self.tp.query("SELECT * FROM process WHERE name = '{}'".format(process)))
        return next(self.tp.query("""
            SELECT SUM(dur)
            FROM thread_state
            INNER JOIN thread ON thread.utid = thread_state.utid
            WHERE thread.upid = {} AND thread_state.state = 'Running' {}
        """.format(process.upid, "AND {}".format(filter) if filter else ""))).__dict__["SUM(dur)"]
    
    def total_cpu_time(self, process: str) -> int:
        return self.__get_cpu_time(process)

    def gc_cpu_time(self, process: str) -> int:
        return self.__get_cpu_time(process, "thread.name = '{}'".format(LBO.GC_THREAD))

    def cpufreq(self):
        return self.tp.query("""
            SELECT ts, value, cpu
            FROM counter
            INNER JOIN cpu_counter_track ON counter.track_id=cpu_counter_track.id
            WHERE cpu_counter_track.name='cpufreq'
        """).as_pandas_dataframe()

In [ ]:
lbo = LBO("example.perfetto-trace")
lbo.gc_cpu_time("com.google.android.apps.maps")/lbo.total_cpu_time("com.google.android.apps.maps")